[← Volver al índice del curso](../../../INDICE_CURSO.md) · [Guía del tema 03](README.md)

# OpenMP: tareas, dependencias y granularidad

**Tema:** 03 · **Sesiones:** 14, 15 · **Edición:** 1.0.2026

**Pregunta guía:** ¿Cuándo un DAG de tareas expone paralelismo suficiente para compensar el costo de creación y sincronización?


## Resultados de aprendizaje

- Representar tareas y dependencias.
- Distinguir task, taskgroup y taskwait.
- Elegir un corte de granularidad mediante medición.


## Modelo conceptual

Las tareas expresan trabajo potencialmente diferido y ejecutado por cualquier hilo del equipo.

Las dependencias se asocian a regiones de almacenamiento y forman un DAG.

Una recursión fina puede producir más overhead que trabajo; un cutoff conserva trabajo secuencial en hojas pequeñas.


In [ ]:
from pathlib import Path

def find_repository(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "INDICE_CURSO.md").is_file():
            return candidate
    raise RuntimeError("No se encontró la raíz del repositorio")

ROOT = find_repository(Path.cwd())
TOPIC = "03"
NOTEBOOK = "03_openmp/03_tareas_rendimiento.ipynb"
assert (ROOT / "curso" / "notebooks" / "03_openmp" / "README.md").is_file()
print(f"Repositorio: {ROOT}")
print(f"Notebook: {NOTEBOOK}")


## Camino crítico de tareas

Se calcula el tiempo mínimo ideal de un DAG de bloques.


In [ ]:
duration = {"A": 3, "B": 5, "C": 2, "D": 4, "E": 1}
deps = {"A": [], "B": [], "C": ["A"], "D": ["A", "B"], "E": ["C", "D"]}
finish = {}
for task in duration:
    finish[task] = duration[task] + max((finish[p] for p in deps[task]), default=0)
work, span = sum(duration.values()), max(finish.values())
assert (work, span) == (15, 10)
print({"work": work, "span": span, "ideal_parallelism": work/span})


**Interpretación.** El span revela si más hilos pueden ayudar antes de considerar overhead y ancho de banda.


## Modelo de cutoff

Se estima cuándo el trabajo por tarea supera un overhead de creación medido.


In [ ]:
overhead_us = 3.2
cost_per_item_us = 0.08
candidates = (8, 16, 32, 64, 128, 256)
for items in candidates:
    useful = items * cost_per_item_us
    ratio = useful / overhead_us
    print(f"items={items:3} trabajo={useful:5.2f}us trabajo/overhead={ratio:4.1f}")
cutoff = next(items for items in candidates if items * cost_per_item_us >= 5 * overhead_us)
assert cutoff == 256
print("cutoff inicial:", cutoff)


**Interpretación.** El factor cinco es una hipótesis de partida; el cutoff final se obtiene en el hardware objetivo.


## Práctica reproducible

1. Dibujar dependencias de mergesort o stencil.
2. Medir número de tareas y tiempo para varios cutoffs.
3. Comparar con una versión `parallel for` cuando la estructura lo permita.


## Errores frecuentes

- Crear una región paralela por llamada recursiva.
- Omitir `single` al generar el DAG.
- Medir solo un tamaño y declarar un cutoff universal.

## Criterios de aceptación

- DAG sin carreras ni dependencias faltantes.
- Cutoff justificado con curva.
- Resultado comparado con versión serial.


## Referencias y material relacionado

- [Planeación OpenMP](../../../docs/PLANEACION_CURSO.md)
- [Ejemplos OpenMP](../../../openmp/)


[← Volver al índice del curso](../../../INDICE_CURSO.md) · [Continuar desde la guía del tema 03](README.md)
